## Executive Summary

**Purpose:** Size the parts around the gears so the whole stage can be made: the sun-to-shaft joint, the planet bearings, the planet pins, the carrier, the ring fixing, and the axial stack that has to fit the envelope length.

**What it does:** `size_components` takes a `PlanetarySet`, its ISO 6336 rating, and a `ComponentInputs` describing the envelope and the input shaft, and returns every component dimension with its check. `rim_thicknesses` gives the rims under the tooth roots (sun over its key groove, planet over its bearing bore, ring outside its roots), which `gears.rating` needs for the rim factor $Y_B$.

**Arrangement assumed:** double-plate (straddle) carrier with $N$ posts between the planets; the output is taken from one plate. Each planet runs on a needle roller cage assembly placed directly on a hardened pin and in the hardened planet bore. The sun is joined to the input shaft by a key (DIN 6885-1), a spline (DIN 5480), a press or shrink fit or a retaining adhesive, or it is cut directly on the shaft. The ring gear is the outer wall of the stage and is clamped between housing parts by through bolts, or pressed and pinned into a housing when the rim is too thin for bolt holes.

## How It Fits Into the Gear Design Workflow

**Upstream:** `gears.layout.PlanetarySet` (centre distance, gear diameters), `gears.rating.PlanetaryRating` (torques and tooth forces).

**Downstream:** `gears.manufacturing` (drawing data: fits and tolerances of the pin, bore and carrier holes), `gears.design`.

In [ ]:
#| default_exp gears.components

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass, field

from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import PlanetaryRating
from emachines.gears.manufacturing import iso286_limits

In [ ]:
#| hide
# drawing helpers for the explanatory figures (not exported)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc, Polygon, Wedge, Rectangle
from emachines.gears import layout as _lay, rating as _rat, components as _cmp   # exported modules, so figures can sit ahead of the definitions

_PS = _lay.PlanetarySet.build(18, 27, 72, m=2.0, n_planets=3, b=20)
_CI = _cmp.ComponentInputs(envelope_d=170, shaft_d=16, envelope_length=50)
_RT = _rat.rate_planetary(_PS, _rat.RatingInputs(torque_out=150.0, n_in=1500.0), rims=_cmp.rim_thicknesses(_PS, _CI))
_CO = _cmp.size_components(_PS, _RT, _CI, life_h=10_000, n_in=1500.0)
from emachines.gears.manufacturing import gear_outline, iso286_limits as _iso

INK, BLUE, RED, GREEN = "#222222", "#1f5fa8", "#b8322a", "#2a8a4a"
C_SUN, C_PLANET, C_RING, C_STEEL = "#d8e8d0", "#f6e7c8", "#dfe7f1", "#bdbdbd"

def _rot(P, a):
    c, s_ = np.cos(a), np.sin(a)
    return np.asarray(P) @ np.array([[c, s_], [-s_, c]])

def _label(ax, text, xy, xytext, color=INK, ha="left", fs=9):
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=fs, color=color, ha=ha, va="center",
                arrowprops=dict(arrowstyle="-", color=color, lw=0.7, shrinkA=0, shrinkB=0))

def _dim(ax, p0, p1, text="", off=(0, 0), color=BLUE, fs=9, ha="center"):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="<->", color=color, lw=0.8, shrinkA=0, shrinkB=0))
    if text:
        ax.text((p0[0] + p1[0]) / 2 + off[0], (p0[1] + p1[1]) / 2 + off[1], text, color=color, fontsize=fs,
                ha=ha, va="center", bbox=dict(fc="white", ec="none", pad=0.6, alpha=0.85))

def _circ(ax, r, style="--", color=BLUE, lw=0.7, c=(0, 0)):
    ax.add_patch(Circle(c, r, fill=False, ls=style, ec=color, lw=lw))

def _set_drawing(ps, ax, turn=np.pi / 2):
    "Draw sun, planets and ring of a PlanetarySet in mesh, turned so planet 1 is at angle `turn`."
    rho = np.pi * ps.planet.z / ps.ring.z
    ax.add_patch(Polygon(_rot(gear_outline(ps.ring), rho + turn), closed=True, fc="white", ec=INK, lw=0.7))
    for k in range(ps.n_planets):
        th = 2 * np.pi * k / ps.n_planets
        psi = th * (1 + ps.sun.z / ps.planet.z) + np.pi - np.pi / ps.planet.z
        c = ps.a_w * np.array([np.cos(th), np.sin(th)])
        ax.add_patch(Polygon(_rot(_rot(gear_outline(ps.planet), psi) + c, turn), closed=True, fc=C_PLANET, ec=INK, lw=0.7))
    ax.add_patch(Polygon(_rot(gear_outline(ps.sun), turn), closed=True, fc=C_SUN, ec=INK, lw=0.7))

## Sun–Shaft Joint: Theory

The joint between the input shaft and the sun decides whether there is backlash between motor and sun, how small the sun can be (a bore and a groove need room under the teeth), and whether the sun can float radially to share the load between the planets. Five options are offered (`ComponentInputs.hub_joint`):

| `hub_joint` | Joint | Backlash | Sun can float | Smallest sun | Take apart |
|---|---|---|---|---|---|
| `key` | parallel key, DIN 6885-1 | small (key side clearance) | yes, with a loose bore | largest: bore + groove + rim | yes |
| `spline` | involute spline, DIN 5480 | `sliding`: yes; `tight`: none | `sliding` only | bore + spline depth + rim | yes |
| `press` | press or shrink fit | none | no | bore + rim | hard |
| `adhesive` | retaining compound on a slip fit | none | no | bore + rim | hard (heat) |
| `integral` | sun cut on the shaft | none | no | smallest: no bore | – |

A joint without backlash also fixes the sun radially, so the load-sharing factor $K_\gamma$ of a fixed sun applies (`RatingInputs.floating_sun = False`); a floating sun needs a key or a sliding spline.

In [ ]:
#| echo: false
def _joints_figure():
    ps = _PS
    sun, d = ps.sun, _CI.shaft_d
    P = _rot(gear_outline(sun), np.pi / 2 + np.pi / sun.z)
    fig, axs = plt.subplots(1, 5, figsize=(15, 4.0))
    titles = [("parallel key", "small backlash (key side clearance)", RED),
              ("involute spline", "sliding fit: backlash; tight fit: none", RED),
              ("press or shrink fit", "no backlash", GREEN),
              ("retaining adhesive", "no backlash", GREEN),
              ("sun cut on the shaft", "no joint, no backlash", GREEN)]
    for ax, (t, bl, col) in zip(axs, titles):
        ax.add_patch(Polygon(P, closed=True, fc=C_SUN, ec=INK, lw=0.7))
        ax.set_xlim(-22, 22); ax.set_ylim(-22, 22); ax.set_aspect("equal"); ax.axis("off")
        ax.set_title(t, fontsize=10)
        ax.text(0, -25, bl, ha="center", va="top", fontsize=8.5, color=col)
    # key
    ax = axs[0]
    k = _cmp.ParallelKey.for_shaft(d)
    ax.add_patch(Circle((0, 0), d / 2 + 0.05, fc="white", ec=INK, lw=0.6))
    ax.add_patch(Rectangle((-k.b / 2, d / 2 - 0.3), k.b, k.t2 + 0.3, fc="white", ec=INK, lw=0.5))
    ax.add_patch(Circle((0, 0), d / 2, fc=C_STEEL, ec=INK, lw=0.7))
    ax.add_patch(Rectangle((-k.b / 2 + 0.25, d / 2 - k.t1), k.b - 0.5, k.h, fc="#888", ec=INK, lw=0.6))
    # spline: shaft and hub teeth (DIN 5480 W16 × 1 style, drawn as trapezoids)
    ax = axs[1]
    m_s = 1.0
    g = _cmp.spline_geometry(d, m_s)
    z = int(g["z"])
    r_hub_root, r_tip_hub = d / 2 + 0.1 * m_s, d / 2 - 0.9 * m_s
    ph = np.linspace(0, 2 * np.pi, 4 * z + 1)
    rr = np.where((np.arange(4 * z + 1) // 2) % 2 == 0, r_hub_root, r_tip_hub)
    hole = np.c_[rr * np.cos(ph + np.pi / (2 * z)), rr * np.sin(ph + np.pi / (2 * z))]
    ax.add_patch(Polygon(hole, closed=True, fc="white", ec=INK, lw=0.5))
    rs = np.where((np.arange(4 * z + 1) // 2) % 2 == 0, d / 2 - 0.1 * m_s, d / 2 - 1.1 * m_s)
    shaft = np.c_[rs * np.cos(ph + np.pi / (2 * z) + 2 * np.pi / (2 * z)), rs * np.sin(ph + np.pi / (2 * z) + 2 * np.pi / (2 * z))]
    ax.add_patch(Polygon(shaft, closed=True, fc=C_STEEL, ec=INK, lw=0.6))
    # press fit: contact pressure on the joint
    ax = axs[2]
    ax.add_patch(Circle((0, 0), d / 2, fc=C_STEEL, ec=INK, lw=0.7))
    ax.add_patch(Circle((0, 0), d / 2, fill=False, ec=RED, lw=2.2))
    for a in np.linspace(0, 2 * np.pi, 12, endpoint=False):
        u = np.array([np.cos(a), np.sin(a)])
        ax.annotate("", xy=u * (d / 2 - 2.6), xytext=u * (d / 2 - 0.4), arrowprops=dict(arrowstyle="-|>", color=RED, lw=0.7))
    ax.text(0, 0, "$p$", color=RED, ha="center", va="center", fontsize=11)
    # adhesive: thin layer (exaggerated)
    ax = axs[3]
    ax.add_patch(Circle((0, 0), d / 2 + 0.3, fc="#f2c94c", ec=INK, lw=0.5))
    ax.add_patch(Circle((0, 0), d / 2 - 0.3, fc=C_STEEL, ec=INK, lw=0.7))
    _label(ax, "compound\nin the gap", (d / 2 * 0.7, d / 2 * 0.7), (8, 19), fs=8)
    # integral
    ax = axs[4]
    ax.add_patch(Circle((0, 0), d / 2, fill=False, ec="#888888", lw=0.8, ls="--"))
    ax.text(0, 0, f"shaft Ø{d:g}\nbeyond the sun\n(dashed)", ha="center", va="center", fontsize=7.5, color="#888888")
    plt.tight_layout()
_joints_figure()

### Parallel key (DIN 6885-1, form A)

The key size (width $b$, height $h$, shaft groove $t_1$, hub groove $t_2$) follows from the shaft diameter. The hub groove depth $t_2$ sets the rim left under the sun teeth:

In [ ]:
#| echo: false
def _key_figure():
    ps, hub = _PS, _CO.sun_hub
    d, b, h, t1, t2 = _CI.shaft_d, hub["key_b"], hub["key_h"], hub["t1"], hub["t2"]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.add_patch(Polygon(_rot(gear_outline(ps.sun), np.pi / 2 + np.pi / ps.sun.z), closed=True, fc=C_SUN, ec=INK, lw=0.9))
    # hub groove (cut into the sun bore) and shaft groove
    ax.add_patch(Circle((0, 0), d / 2, fc=C_STEEL, ec=INK, lw=0.8))
    ax.add_patch(Rectangle((-b / 2, d / 2 - 0.3), b, t2 + 0.3, fc="white", ec=INK, lw=0.6))
    ax.add_patch(Rectangle((-b / 2, d / 2 - t1), b, h, fc="#888", ec=INK, lw=0.7))
    ax.add_patch(Circle((0, 0), ps.sun.df / 2, fill=False, ec=BLUE, ls="--", lw=0.8))
    x0 = 24
    chain = [(0, d / 2, f"$d_{{sh}}/2$ = {d / 2:g}", 24), (d / 2 - t1, d / 2, f"$t_1$ = {t1:g}: shaft groove", 30),
             (d / 2, d / 2 + t2, f"$t_2$ = {t2:g}: hub groove", 24),
             (d / 2 + t2, ps.sun.df / 2, f"sun rim = {hub['rim']:.2f} (needs $\\geq 0.5\\,h_t$; $Y_B > 1$ below $1.2\\,h_t$)", 30)]
    for r0, r1, txt, xx in chain:
        _dim(ax, (xx, r0), (xx, r1), color=BLUE)
        ax.plot([0, xx + 1], [r1, r1], color=INK, lw=0.4, ls=":")
        ax.text(xx + 1.5, (r0 + r1) / 2, txt, color=BLUE, fontsize=9, va="center")
    _dim(ax, (-b / 2, -2.5 + d / 2 - t1), (b / 2, -2.5 + d / 2 - t1), "b", off=(0, -1.6), color=INK)
    _label(ax, f"key {hub['key']}: width b = {b:g}, height h = {h:g}", (-b / 2, d / 2), (-24, 18), ha="right")
    _label(ax, f"shaft Ø{d:g} (k6), sun bore H7", (-d / 2 * 0.6, -d / 2 * 0.3), (-24, -6), ha="right")
    _label(ax, "sun root circle $d_{f,s}$", (-ps.sun.df / 2 * 0.7, -ps.sun.df / 2 * 0.71), (-24, -18), color=BLUE, ha="right")
    _label(ax, "flank pressure $p$ acts on the hub\nside of the key, depth $h - t_1$", (b / 2, d / 2 + 0.8), (24, -8))
    ax.set_xlim(-60, 80); ax.set_ylim(-24, 26); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Sun on its shaft with a DIN 6885-1 key (example: Ø{d:g} shaft, sun z = {ps.sun.z}, m = {ps.sun.m:g})", fontsize=10)
    plt.tight_layout()
_key_figure()

$$s_{R,s} = \frac{d_{f,s}}{2} - \frac{d_{sh}}{2} - t_2 \ \ge\ 0.5\,h_t\quad(h_t = (h_{aP}^* + h_{fP}^*)\,m;\ \text{below } 1.2\,h_t \text{ the rating applies } Y_B > 1).$$

The key length is the longest DIN 6885 standard length not above the hub width, $l \le b + 2s_w$. The key transmits the sun torque, with the application factor, by flank pressure on the hub side:

$$p = \frac{2000\,K_A T_s}{d_{sh}\,(h - t_1)\,l_{tr}} \le p_{allow},\qquad l_{tr} = l - b\ \text{(rounded ends)},$$

with $p_{allow} \approx 0.9\,R_e$ of the weaker part for steady load; 150 MPa is a safe default for a hardened sun on a steel shaft. A sun that is case carburized needs its key groove cut before heat treatment (or the groove masked during carburizing), because the hardened case is too hard to broach.

A key always has some side clearance in its grooves (P9 or JS9 hub groove), so it leaves a small backlash between motor and sun.

### Involute spline (DIN 5480)

For a spline of nominal diameter $d$ and module $m_{sp}$ (30° pressure angle) the code uses

$$z = \Big\lfloor \frac{d - 1.1\,m_{sp}}{m_{sp}} \Big\rfloor,\qquad d_m = d - 1.1\,m_{sp},\qquad h_w = 0.9\,m_{sp},$$

and the mean flank pressure over the joint length $L$ (the sun width), with only a share $\varphi = 0.75$ of the teeth carrying because of pitch errors (DIN 5466-1 practice):

$$p = \frac{2000\,K_A T_s}{d_m\,h_w\,L\,z\,\varphi} \le p_{allow}.$$

The hub spline root lies about $t_2 = (0.55 + 1.1)\,m_{sp} = 1.65\,m_{sp}$ outside the nominal diameter, so the rim is $s_{R,s} = d_{f,s}/2 - d/2 - t_2$. The fit decides the backlash: `sliding` (9H/8f) has flank clearance, so backlash, but lets a floating sun centre itself; `tight` (an interference fit class such as 9H/8k) has none.

### Press or Shrink Fit

The sun (hub, outer diameter taken as the root circle $D_o = d_{f,s}$, the teeth ignored) is shrunk onto a solid shaft of diameter $d$ over the length $L$. Both are steel ($E$ = 210 GPa). The interference of a fit is

$$\delta_{min} = ei_{shaft} - ES_{hole} - \Delta,\qquad \delta_{max} = es_{shaft} - EI_{hole} - \Delta,\qquad \Delta = 0.8\,(Rz_{shaft} + Rz_{hole})$$

where $\Delta$ is the roughness smoothed away on assembly (DIN 7190). Lamé's thick-cylinder solution with $Q = d/D_o$ gives the joint pressure

$$p = \frac{E\,\delta\,(1 - Q^2)}{2d}.$$

The smallest pressure must carry the torque with the slip safety $S$ (default 1.5) and friction $\mu$ (0.12 for a shrink fit):

$$T_{slip} = \mu\,p_{min}\,\pi d L\,\frac{d}{2} \ \ge\ S\,K_A T_s,$$

and the largest must not overstress the bore (Tresca at the bore, against $\sigma_{allow}$ = 500 MPa, about $R_{p0.2}/1.4$ of a 20MnCr5 core):

$$\sigma_{v} = \frac{2p_{max}}{1 - Q^2} \le \sigma_{allow}.$$

The fit is the lightest of H6/r5, H6/s5, H6/u5 and H7/u6 that meets both (a fit can also be given). The hole is H6 because the H7 spread is too wide for a small shaft: the lower end gives no grip and the upper end overstresses the bore.

In [ ]:
#| echo: false
def _press_figure():
    ps = _PS
    d, Do, L = _CI.shaft_d, ps.sun.df, ps.sun.b
    E, mu, Rz = _cmp.E_STEEL, 0.12, 2.0
    T_req = 1.5 * 1.25 * _RT.T_sun
    sel = _cmp.select_press_fit(d, Do, L, T_req, 500.0, mu, Rz)
    Q = d / Do
    fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.8))
    # (a) Lamé stresses at p_max
    ax = axs[0]
    p = sel.p_max
    r = np.linspace(d / 2, Do / 2, 100)
    a2, b2 = (d / 2) ** 2, (Do / 2) ** 2
    s_t = p * a2 / (b2 - a2) * (1 + b2 / r ** 2)
    s_r = p * a2 / (b2 - a2) * (1 - b2 / r ** 2)
    ax.plot(r, s_t, color=RED, lw=1.3, label="hub, hoop $\\sigma_t$")
    ax.plot(r, s_r, color=BLUE, lw=1.3, label="hub, radial $\\sigma_r$")
    ax.plot([0, d / 2], [-p, -p], color="#888888", lw=1.3, label="solid shaft, $\\sigma_r = \\sigma_t = -p$")
    ax.axhline(0, color=INK, lw=0.6)
    ax.axvline(d / 2, color=INK, lw=0.5, ls=":"); ax.axvline(Do / 2, color=INK, lw=0.5, ls=":")
    ax.text(d / 2, ax.get_ylim()[0] if False else -p - 25, "joint\n$d/2$", ha="center", va="top", fontsize=8.5)
    ax.text(Do / 2, -p - 25, "sun root\n$d_f/2 = D_o/2$", ha="center", va="top", fontsize=8.5)
    ax.annotate(f"$\\sigma_t = p\\,\\frac{{1+Q^2}}{{1-Q^2}}$ = {s_t[0]:.0f}", xy=(d / 2, s_t[0]), xytext=(d / 2 + 1.5, s_t[0] + 40),
                fontsize=9, color=RED, arrowprops=dict(arrowstyle="-", color=RED, lw=0.6))
    ax.annotate(f"$\\sigma_t = \\frac{{2pQ^2}}{{1-Q^2}}$ = {s_t[-1]:.0f}\n(adds to the root bending stress)", xy=(Do / 2, s_t[-1]),
                xytext=(9.0, 15), fontsize=9, color=RED, arrowprops=dict(arrowstyle="-", color=RED, lw=0.6))
    ax.set_xlim(0, Do / 2 + 1.5); ax.set_ylim(-p - 70, s_t[0] + 120)
    ax.set_xlabel("radius [mm]"); ax.set_ylabel("stress [MPa]")
    ax.legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False)
    ax.set_title(f"(a) stresses at the largest interference ({sel.fit}, $p_{{max}}$ = {p:.0f} MPa)", fontsize=10)
    # (b) interference windows of the candidate fits
    ax = axs[1]
    loss = 0.8 * 2 * Rz
    k = E * (1 - Q ** 2) / (2 * d) / 1000                       # MPa per µm
    d_req = 2000 * T_req / (mu * np.pi * d ** 2 * L) / k + loss
    d_all = 500.0 * d / E * 1000 + loss
    for i, f in enumerate(_cmp.PRESS_FITS):
        h, s_ = f.split("/")
        ES, EI = (1000 * v for v in _iso(d, h)); es, ei = (1000 * v for v in _iso(d, s_))
        lo, hi = ei - ES, es - EI
        ax.add_patch(Rectangle((i - 0.25, lo), 0.5, hi - lo, fc=C_PLANET if f != sel.fit else "#cfe8d6", ec=INK, lw=0.7))
        ax.text(i, hi + 1.5, f, ha="center", fontsize=9, fontweight="bold" if f == sel.fit else "normal")
    ax.axhline(d_req, color=GREEN, lw=1.0, ls="--")
    ax.text(len(_cmp.PRESS_FITS) - 0.35, d_req, f"needed for torque:\n$\\delta_{{min}} \\geq$ {d_req:.0f} µm", color=GREEN, fontsize=8.5, va="center", ha="left")
    ax.axhline(d_all, color=RED, lw=1.0, ls="--")
    ax.text(len(_cmp.PRESS_FITS) - 0.35, d_all, f"allowed by bore stress:\n$\\delta_{{max}} \\leq$ {d_all:.0f} µm", color=RED, fontsize=8.5, va="center", ha="left")
    ax.set_xlim(-0.6, len(_cmp.PRESS_FITS) + 1.2); ax.set_ylim(0, max(d_all, 60) + 15)
    ax.set_xticks([]); ax.set_ylabel("geometric interference [µm]")
    [ax.spines[s].set_visible(False) for s in ("top", "right")]
    ax.set_title(f"(b) interference of each fit at Ø{d:g} (chosen: {sel.fit})", fontsize=10)
    plt.tight_layout()
_press_figure()

The shrink fit has three side effects that are reported:

- **Hoop stress at the tooth root**, $\sigma_t(D_o) = 2p_{max}Q^2/(1 - Q^2)$. It is a steady tensile stress under the teeth that lowers the bending margin. The ISO 6336 rating does not include it, so keep $S_F$ well above its minimum with this joint.
- **Sun growth.** The root circle grows by $\Delta d = 2p_{max}Q^2 D_o / \big(E(1 - Q^2)\big)$, which takes backlash away from the sun–planet mesh like a longer centre distance: $\Delta j \approx \Delta d\,\sin\alpha_w$. The check requires $\Delta j \le 0.5\,j_{bn,min}$.
- **Assembly.** Shrinking needs the sun heated to $T = 20 + (\delta_{max,geom} + 0.001\,d)/(\alpha\,d)$ °C ($\alpha = 11.5\cdot10^{-6}$/K, 0.001 d joining clearance). A case-hardened sun must stay below about 150 °C, so a small sun usually needs the shaft cooled too (dry ice or liquid nitrogen). Pressing in cold takes $F = \mu_{press}\,p_{max}\,\pi d L$ ($\mu_{press}$ = 0.08); never press the sun onto a motor shaft through the motor bearings.

### Retaining Adhesive

An anaerobic retaining compound fills the gap of a slip or transition fit (H7/k6) and carries the torque in shear:

$$T_{cap} = \tau_{allow}\,\pi d L\,\frac{d}{2} \ \ge\ S\,K_A T_s,$$

with $\tau_{allow}$ = 8 MPa as a design value (the catalogue strength of 15–25 MPa reduced for the gap, temperature and ageing). It has no backlash, needs no heating, and can be combined with a light press fit. It softens above about 150 °C, which is also how it is taken apart.

### Sun Cut on the Shaft

With the teeth hobbed (or shaped) and ground directly on the input shaft there is no joint at all: no backlash, no bore, no rim limit, so the sun can be as small as its tooth count allows (in `gears.layout.module_limits` the bore is taken as zero). The motor is then connected to this shaft by a clamp hub or a torsionally stiff coupling. `shaft_d` now means the shaft beside the sun. The checks are the torsion in the shaft at the root diameter,

$$\tau = \frac{16\cdot 1000\,K_A T_s}{\pi d_{f,s}^3} \le \tau_{allow},$$

and the room for the hob. A shaft below the root circle lets the hob run out freely. A shoulder that stands $c = d_{sh}/2 - (d_{f,s}/2 - 0.25\,m)$ above the root needs an axial gap from the gear face, so that a hob of tip diameter $d_{a0}$ (default 40 mm) can finish the full depth at the face:

$$L_{ro} = \sqrt{c\,(d_{a0} - c)} + 1\ \text{mm}.$$

In [ ]:
#| echo: false
def _integral_figure():
    s = _PS.sun
    ra, rf, b = s.da / 2, s.df / 2, s.b
    hob_d = 40.0
    fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.8))
    for ax, rj, title in ((axs[0], rf - 2.5, "(a) shaft beside the sun below the root circle"),
                          (axs[1], ra + 3.0, "(b) shoulder above the root circle: leave a hob run-out")):
        L_ro = _cmp.hob_runout(rj - rf, hob_d)
        if rj < rf:
            ax.add_patch(Rectangle((-30, -rj), 30, 2 * rj, fc=C_STEEL, ec=INK, lw=0.7))
        else:
            ax.add_patch(Rectangle((-L_ro, -8), L_ro, 16, fc=C_STEEL, ec=INK, lw=0.7))
            ax.add_patch(Rectangle((-L_ro - 8, -rj), 8, 2 * rj, fc=C_STEEL, ec=INK, lw=0.7))
        ax.add_patch(Rectangle((0, -ra), b, 2 * ra, fc=C_SUN, ec=INK, lw=0.8))
        ax.add_patch(Rectangle((0, -rf), b, 2 * rf, fc="#c3dbbb", ec="none"))
        ax.add_patch(Rectangle((b, -8), 22, 16, fc=C_STEEL, ec=INK, lw=0.7))
        ax.plot([-40, b + 24], [0, 0], color=RED, lw=0.5, ls="-.")
        ax.plot([-40, b + 22], [rf, rf], color=BLUE, lw=0.6, ls="--")
        ax.text(b + 23, rf, "root circle $d_f$", color=BLUE, fontsize=9, va="center")
        ax.text(b / 2, -2, "sun teeth\ncut and ground\non the shaft", ha="center", va="center", fontsize=8.5)
        ax.text(b + 11, -10, "to the motor:\nclamp hub or\nstiff coupling", ha="center", va="top", fontsize=8)
        # hob centred on the gear face, cutting to full depth there
        yc = rf + hob_d / 2
        ax.add_patch(Circle((0, yc), hob_d / 2, fill=False, ec=GREEN, lw=0.8, ls="--"))
        ax.text(0, yc + 3, f"hob Ø{hob_d:g}\nat the end of its cut", color=GREEN, ha="center", fontsize=8.5)
        if rj > rf:
            _dim(ax, (-L_ro, rj + 3), (0, rj + 3), "", color=RED)
            ax.text(-L_ro / 2, rj + 4.5, f"$L_{{ro}}$ = {L_ro:.1f} mm", color=RED, ha="center", fontsize=9)
        else:
            ax.text(-28, rj + 3, "the hob passes over\nthe smaller shaft", fontsize=8.5, color=GREEN, ha="center", va="bottom")
        ax.set_xlim(-42, b + 34); ax.set_ylim(-ra - 8, rf + hob_d + 3); ax.set_aspect("equal"); ax.axis("off")
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
_integral_figure()

In [ ]:
#| export
# DIN 6885-1 parallel keys: (d_min, d_max] → (b, h, t1 shaft groove, t2 hub groove), mm
DIN6885_KEYS = [
    (6, 8, 2, 2, 1.2, 1.0), (8, 10, 3, 3, 1.8, 1.4), (10, 12, 4, 4, 2.5, 1.8), (12, 17, 5, 5, 3.0, 2.3),
    (17, 22, 6, 6, 3.5, 2.8), (22, 30, 8, 7, 4.0, 3.3), (30, 38, 10, 8, 5.0, 3.3), (38, 44, 12, 8, 5.0, 3.3),
    (44, 50, 14, 9, 5.5, 3.8), (50, 58, 16, 10, 6.0, 4.3), (58, 65, 18, 11, 7.0, 4.4), (65, 75, 20, 12, 7.5, 4.9),
    (75, 85, 22, 14, 9.0, 5.4), (85, 95, 25, 14, 9.0, 5.4), (95, 110, 28, 16, 10.0, 6.4),
]


@dataclass(frozen=True)
class ParallelKey:
    "DIN 6885-1 form A key for one shaft diameter. Lengths in mm."
    d: float
    b: float
    h: float
    t1: float
    t2: float

    @classmethod
    def for_shaft(cls, d: float) -> "ParallelKey":
        for lo, hi, b, h, t1, t2 in DIN6885_KEYS:
            if lo < d <= hi:
                return cls(d, b, h, t1, t2)
        raise ValueError(f"no DIN 6885-1 key listed for d = {d} mm (6 < d ≤ 110)")

    def pressure(self, torque: float, length: float) -> float:
        "Hub-side flank pressure [MPa] for torque [N·m] and key length [mm] (rounded ends)."
        l_tr = length - self.b
        if l_tr <= 0:
            raise ValueError("key shorter than its width")
        return 2000 * torque / (self.d * (self.h - self.t1) * l_tr)

    @staticmethod
    def standard_length(l_max: float) -> float:
        "Longest DIN 6885 standard key length not above l_max [mm]."
        L = [6, 8, 10, 12, 14, 16, 18, 20, 22, 25, 28, 32, 36, 40, 45, 50, 56, 63, 70, 80, 90, 100, 110, 125]
        ok = [x for x in L if x <= l_max + 1e-9]
        if not ok:
            raise ValueError("no standard key length fits")
        return float(ok[-1])

In [ ]:
k14 = ParallelKey.for_shaft(14)
print(k14, f"p = {k14.pressure(10.0, 16):.0f} MPa at 10 N·m, 16 mm key")
assert (k14.b, k14.h, k14.t2) == (5, 5, 2.3) and ParallelKey.standard_length(19) == 18

In [ ]:
#| export
E_STEEL, NU_STEEL, ALPHA_STEEL = 210_000.0, 0.3, 11.5e-6     # MPa, –, 1/K


def spline_geometry(d: float, m: float) -> dict[str, float]:
    "DIN 5480 involute spline on nominal diameter d [mm] with module m: teeth z, mean diameter d_m, bearing height h_w."
    z = int((d - 1.1 * m) // m)
    return {"z": z, "d_m": d - 1.1 * m, "h_w": 0.9 * m}


def spline_pressure(torque: float, d: float, m: float, length: float, phi: float = 0.75) -> float:
    "Mean flank pressure [MPa] of a DIN 5480 spline for torque [N·m] over `length` [mm]; phi: share of teeth carrying."
    g = spline_geometry(d, m)
    return 2000 * torque / (g["d_m"] * g["h_w"] * length * g["z"] * phi)


SPLINE_FITS = {"sliding": ("9H/8f", "clearance: backlash, lets a floating sun self-centre"),
               "tight": ("9H/8k", "flank interference: no backlash, sun fixed radially")}


PRESS_FITS = ("H6/r5", "H6/s5", "H6/u5", "H7/u6")     # tried lightest first


@dataclass(frozen=True)
class PressFit:
    "Interference fit of a solid shaft in a hub (Lamé, equal E and ν). Lengths mm, pressures MPa, torque N·m."
    fit: str
    d: float
    D_o: float
    L: float
    delta_min: float        # effective interference after smoothing [mm]
    delta_max: float
    p_min: float
    p_max: float
    T_slip: float           # torque the fit carries at p_min
    sigma_hub: float        # equivalent stress at the hub bore at p_max (Tresca)
    sigma_root: float       # hoop stress at the hub outer diameter (sun root circle) at p_max
    growth: float           # growth of the hub outer diameter at p_max [mm]
    T_join: float           # hub temperature for shrink assembly [°C]
    F_press: float          # force to press in cold at p_max [N]


def press_fit(d: float, D_o: float, L: float, fit: str, mu: float = 0.12, Rz: float = 2.0,
              mu_press: float = 0.08, E: float = E_STEEL, alpha: float = ALPHA_STEEL) -> PressFit:
    """
    Interference fit `fit` ('H6/s5', 'H7/u6', …) of a solid shaft d in a hub of outer diameter D_o, joint length L
    (see theory). Rz: roughness of each surface [µm], partly smoothed away on assembly.
    """
    hole, shaft = fit.split("/")
    ES, EI = iso286_limits(d, hole)
    es, ei = iso286_limits(d, shaft)
    loss = 0.8 * 2 * Rz / 1000
    dmin, dmax = max(ei - ES - loss, 0.0), max(es - EI - loss, 0.0)
    Q = d / D_o
    k = E * (1 - Q ** 2) / (2 * d)
    p_lo, p_hi = k * dmin, k * dmax
    return PressFit(fit, d, D_o, L, dmin, dmax, p_lo, p_hi,
                    T_slip=mu * p_lo * math.pi * d * L * d / 2 / 1000,
                    sigma_hub=2 * p_hi / (1 - Q ** 2),
                    sigma_root=2 * p_hi * Q ** 2 / (1 - Q ** 2),
                    growth=2 * p_hi * Q ** 2 * D_o / (E * (1 - Q ** 2)),
                    T_join=20 + ((es - EI) + 0.001 * d) / (alpha * d),
                    F_press=mu_press * p_hi * math.pi * d * L)


def select_press_fit(d: float, D_o: float, L: float, T_req: float, sigma_allow: float, mu: float = 0.12,
                     Rz: float = 2.0, fits: tuple[str, ...] = PRESS_FITS) -> PressFit:
    "Lightest fit that carries T_req [N·m] without exceeding sigma_allow at the hub bore; else the strongest within stress, else the lightest."
    trials = [press_fit(d, D_o, L, f, mu, Rz) for f in fits]
    for t in trials:
        if t.T_slip >= T_req and t.sigma_hub <= sigma_allow:
            return t
    within = [t for t in trials if t.sigma_hub <= sigma_allow]
    return within[-1] if within else trials[0]


def adhesive_torque(d: float, L: float, tau_allow: float) -> float:
    "Torque [N·m] a retaining-compound joint of diameter d and length L [mm] carries at shear stress tau_allow [MPa]."
    return tau_allow * math.pi * d * L * d / 2 / 1000


def hob_runout(c: float, hob_d: float) -> float:
    """
    Axial gap [mm] needed between the gear face and a shoulder that stands c [mm] above the root circle, so that a
    hob of tip diameter hob_d can cut the full depth up to the face (plus 1 mm). 0 if the shoulder is below the root.
    """
    if c <= 0:
        return 0.0
    c = min(c, hob_d / 2)
    return math.sqrt(c * (hob_d - c)) + 1.0

In [ ]:
pf = press_fit(16, 31.0, 20, "H6/s5")
print(f"H6/s5 Ø16 in Ø31 hub: δ = {pf.delta_min * 1000:.1f}…{pf.delta_max * 1000:.1f} µm, p = {pf.p_min:.0f}…{pf.p_max:.0f} MPa, "
      f"T_slip = {pf.T_slip:.1f} N·m, σ_hub = {pf.sigma_hub:.0f} MPa, growth {pf.growth * 1000:.1f} µm, join at {pf.T_join:.0f} °C")
print(f"spline W16 × 1: {spline_geometry(16, 1.0)}, p = {spline_pressure(10, 16, 1.0, 20):.0f} MPa at 10 N·m",
      f"adhesive Ø16 × 20 at 8 MPa: {adhesive_torque(16, 20, 8):.1f} N·m, hob run-out for a shoulder 4.5 mm above the root: {hob_runout(4.5, 40):.1f} mm")
assert pf.p_max > pf.p_min > 0 and pf.T_slip > 0
sel = select_press_fit(16, 31, 20, 5.0, 500)
assert sel.T_slip >= 5.0 and sel.sigma_hub <= 500 and select_press_fit(16, 31, 20, 1e6, 500).sigma_hub <= 500

## Planet Bearing: Theory

The planet runs on a needle roller cage directly on the pin: the pin is the inner raceway ($F_w$), the planet bore the outer raceway ($E_w$), with rollers of diameter $D_w$ in a cage of width $B_c$.

In [ ]:
#| echo: false
def _bearing_figure():
    ps, br, pin = _PS, _CO.bearing, _CO.pin
    Fw, Ew, Dw, Bc, Z = br["Fw"], br["Ew"], br["Dw"], br["Bc"], br["rollers"]
    pl = ps.planet
    fig, (a0, a1) = plt.subplots(1, 2, figsize=(12, 5.4), gridspec_kw=dict(width_ratios=[1.1, 1]))
    # front view
    a0.add_patch(Polygon(gear_outline(pl), closed=True, fc=C_PLANET, ec=INK, lw=0.8))
    a0.add_patch(Circle((0, 0), Ew / 2, fc="white", ec=INK, lw=0.8))
    for j in range(Z):
        a = 2 * np.pi * j / Z
        a0.add_patch(Circle((Fw + Dw) / 2 * np.array([np.cos(a), np.sin(a)]), Dw / 2 * 0.92, fc="#9a9a9a", ec=INK, lw=0.3))
    a0.add_patch(Circle((0, 0), Fw / 2, fc=C_STEEL, ec=INK, lw=0.8))
    a0.add_patch(Circle((0, 0), pl.df / 2, fill=False, ec=BLUE, ls="--", lw=0.7))
    _label(a0, f"pin, $F_w$ = Ø{Fw:g} (j5): inner raceway", (Fw / 4, -Fw / 4), (32, -28))
    _label(a0, f"needle rollers $D_w$ = {Dw:g}, Z = {Z}", ((Fw + Dw) / 2 * np.cos(0.8), (Fw + Dw) / 2 * np.sin(0.8)), (32, 30))
    _label(a0, f"planet bore $E_w = F_w + 2D_w$ = Ø{Ew:g} (G6):\nouter raceway", (Ew / 2 * np.cos(2.4), Ew / 2 * np.sin(2.4)), (-32, 30), ha="right")
    _label(a0, f"planet root circle; rim over the bore\n$(d_{{f,p}} - E_w)/2$ = {br['planet_rim']:.1f}", (-pl.df / 2 * 0.8, -pl.df / 2 * 0.6), (-32, -30), color=BLUE, ha="right")
    a0.set_xlim(-75, 85); a0.set_ylim(-38, 38); a0.set_aspect("equal"); a0.axis("off")
    a0.set_title(f"Planet with a needle cage K {Fw:g} × {Ew:g} × {Bc:g}", fontsize=10)
    # axial half-section: planet, cage, pin
    b, sw, tc = pl.b, _CI.washer_t, _CO.carrier["plate_t"]
    x0 = -(b / 2 + sw + tc)
    a1.add_patch(Rectangle((-b / 2, Ew / 2), b, pl.df / 2 - Ew / 2, fc=C_PLANET, ec=INK, lw=0.8, hatch="////"))
    a1.add_patch(Rectangle((-Bc / 2, Fw / 2), Bc, Dw, fc="#9a9a9a", ec=INK, lw=0.6))
    a1.add_patch(Rectangle((x0, -Fw / 2), 2 * abs(x0), Fw, fc=C_STEEL, ec=INK, lw=0.8))
    for s in (-1, 1):
        a1.add_patch(Rectangle((s * (b / 2) if s > 0 else -b / 2 - sw, Fw / 2), sw, 6, fc="#777", ec=INK, lw=0.5))
        a1.add_patch(Rectangle((s * (b / 2 + sw) if s > 0 else -b / 2 - sw - tc, Fw / 2), tc, 14, fc="#cfe8d6", ec=GREEN, lw=0.8, hatch="xx"))
    a1.plot([x0 - 3, -x0 + 3], [0, 0], color=RED, lw=0.5, ls="-.")
    _dim(a1, (-b / 2, pl.df / 2 + 3), (b / 2, pl.df / 2 + 3), f"face width b = {b:g}", off=(0, 2.2), color=BLUE)
    _dim(a1, (-Bc / 2, -Fw / 2 - 3), (Bc / 2, -Fw / 2 - 3), f"cage width $B_c$ = {Bc:g} (≈ b − 1)", off=(0, -2.2), color=BLUE)
    _label(a1, f"thrust washer $s_w$ = {sw:g}", (b / 2 + sw / 2, Fw / 2 + 5), (22, 22))
    _label(a1, f"carrier plate $t_c$ = {tc:g}", (b / 2 + sw + tc / 2, Fw / 2 + 12), (22, 15), color=GREEN)
    a1.set_xlim(-28, 42); a1.set_ylim(-14, pl.df / 2 + 9); a1.set_aspect("equal"); a1.axis("off")
    a1.set_title("Axial half-section through one planet", fontsize=10)
    plt.tight_layout()
_bearing_figure()

**Load.** The sun and ring push on each planet with the same force $F_{bn}$ along their two lines of action. Their tangential parts add and their radial parts nearly cancel (exactly when $\alpha_{w1} = \alpha_{w2}$), so the pin carries

$$F_{pin} = K_\gamma F_{bn}\sqrt{(\cos\alpha_{w1} + \cos\alpha_{w2})^2 + (\sin\alpha_{w1} - \sin\alpha_{w2})^2} + m_p\,\omega_c^2\,a_w,$$

the last term being the planet's centrifugal force, $m_p$ its mass and $\omega_c$ the carrier speed:

$$m_p = \rho\,\frac{\pi}{4}\left(d_p^2 - E_w^2\right) b,\qquad \omega_c = \frac{2\pi n_c}{60}.$$

The bearing turns at the planet's speed relative to the carrier, $n = \lvert n_{p/c}\rvert$, and the equivalent load is $P = F_{pin}$ (pure radial load).

**Required capacity** for an $L_{10h}$ basic rating life (roller bearing, exponent 10/3):

$$C_{req} = P\left(\frac{60\,n\,L_{10h}}{10^6}\right)^{3/10},\qquad L_{10h} = \frac{10^6}{60\,n}\left(\frac{C_r}{P}\right)^{10/3}.$$

**Space.** The bearing outer raceway is the planet bore; the rim between the planet root and the bore must keep at least $c_p$ tooth depths ($c_p = 1.2$: no rim penalty). That fixes the largest bore,

$$E_{w,max} = \left\lfloor d_{f,p} - 2\,c_p\,(h_{aP}^* + h_{fP}^*)\,m \right\rfloor.$$

A needle roller cage assembly (ISO 3030, type K $F_w \times E_w \times B_c$) has rollers of diameter $D_w$ with $E_w = F_w + 2D_w$, so the pin diameter is $F_w$. The roller size follows the pin size as in the catalogues:

| $F_w$ [mm] | ≤ 8 | ≤ 12 | ≤ 25 | ≤ 40 | ≤ 60 | > 60 |
|---|---|---|---|---|---|---|
| $D_w$ [mm] | 1.5 | 2.0 | 2.5 | 3.0 | 3.5 | 4.0 |

and the cage width is $B_c = \lfloor b - 1 \rfloor$ mm (1 mm shorter than the planet). `size_components` tries bores $E_w$ = 8, 9, 10, … mm up to $E_{w,max}$ and takes the first one that passes all of

$$L_{10h} \ge L_{10h,req},\qquad \sigma_b \le \sigma_{allow},\qquad \frac{10\,\delta}{b} \le 0.01\ \text{mm},\qquad p_{plate} \le p_{allow}$$

(the pin checks are below). A smaller pin keeps the carrier bosses inside the ring.

**Capacity estimate.** Until a catalogue part is chosen, the dynamic load rating of the cage assembly is estimated with the ISO 281 formula for line-contact roller bearings,

$$C_r = b_m f_c (L_{we})^{7/9} Z^{3/4} D_w^{29/27},\qquad b_m = 1.1,$$

with $Z = \lfloor \pi D_{pw}/(1.25\,D_w) \rfloor$ rollers on the pitch circle $D_{pw} = F_w + D_w$, effective roller length $L_{we} = \max(B_c - 3\ \text{mm},\ 0.6\,B_c)$ and $f_c$ from ISO 281 Table 7 as a function of $\gamma = D_w/D_{pw}$ (interpolated linearly in the table). It is within about 20 % of catalogue values; select the real part from the INA, SKF, IKO or NBC catalogue with $C_r \ge C_{req}$ and check its own speed limit.

**Raceway requirements** when rolling directly on pin and bore: 58–64 HRC, case depth ≥ 0.3 mm plus grinding allowance, $R_a \le 0.2\ \mu$m, roundness and cylindricity within IT3–IT4. Normal operating clearance: planet bore **G6**, pin **j5**.

In [ ]:
#| export
_FC_TABLE = [(0.01, 52.1), (0.02, 60.8), (0.03, 66.5), (0.04, 70.7), (0.05, 74.1), (0.06, 76.9), (0.07, 79.2),
             (0.08, 81.2), (0.09, 82.8), (0.10, 84.2), (0.12, 86.4), (0.14, 87.7), (0.16, 88.5), (0.18, 88.8),
             (0.20, 88.7), (0.22, 88.2), (0.24, 87.5), (0.26, 86.4), (0.28, 85.2), (0.30, 83.8)]


def _fc(gamma: float) -> float:
    pts = _FC_TABLE
    if gamma <= pts[0][0]:
        return pts[0][1]
    for (g0, f0), (g1, f1) in zip(pts, pts[1:]):
        if gamma <= g1:
            return f0 + (f1 - f0) * (gamma - g0) / (g1 - g0)
    return pts[-1][1]


def needle_roller_diameter(Fw: float) -> float:
    "Typical roller diameter [mm] of a needle cage assembly for pin diameter Fw [mm]."
    for lim, Dw in ((8, 1.5), (12, 2.0), (25, 2.5), (40, 3.0), (60, 3.5), (1e9, 4.0)):
        if Fw <= lim:
            return Dw
    return 4.0


def needle_cage_rating(Fw: float, Dw: float, Bc: float) -> tuple[float, int]:
    "Estimated dynamic load rating C_r [N] and roller count of a needle cage assembly (ISO 281 formula)."
    Dpw = Fw + Dw
    Z = int(math.pi * Dpw / (1.25 * Dw))
    Lwe = max(Bc - 3.0, 0.6 * Bc)
    return 1.1 * _fc(Dw / Dpw) * Lwe ** (7 / 9) * Z ** 0.75 * Dw ** (29 / 27), Z

In [ ]:
C, Z = needle_cage_rating(20, 3, 17)
print(f"K 20×26×17 estimate: C_r ≈ {C/1000:.1f} kN with {Z} rollers")
assert 15e3 < C < 26e3

## Planet Pin: Theory

The pin is held at both ends in the carrier plates. Treating it as simply supported at mid-plate (span $L = b + 2s_w + t_c$, with face width $b$, thrust washers of thickness $s_w$ and plates of thickness $t_c$) and loaded by $F_{pin}$ spread over the bearing length $B_c$ is conservative for press-fitted ends:

In [ ]:
#| echo: false
def _pin_figure():
    pin, br = _CO.pin, _CO.bearing
    L, Bc, F = pin["span"], br["Bc"], br["F_pin"]
    fig, (a0, a1) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[1.2, 1]))
    a0.plot([-L / 2, L / 2], [0, 0], color=INK, lw=3)
    for s in (-1, 1):
        a0.add_patch(Polygon([[s * L / 2, 0], [s * L / 2 - 1.2, -1.6], [s * L / 2 + 1.2, -1.6]], closed=True, fc="white", ec=INK, lw=0.8))
    for x in np.linspace(-Bc / 2, Bc / 2, 9):
        a0.annotate("", xy=(x, 0.15), xytext=(x, 2.2), arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.0))
    a0.plot([-Bc / 2, Bc / 2], [2.2, 2.2], color=RED, lw=1.0)
    a0.text(0, 2.6, f"$F_{{pin}}$ = {F:.0f} N spread over the bearing length $B_c$ = {Bc:g}", color=RED, ha="center", fontsize=9)
    _dim(a0, (-L / 2, -3.0), (L / 2, -3.0), f"span L = b + 2$s_w$ + $t_c$ = {L:g} (plate mid to plate mid)", off=(0, -0.7), color=BLUE)
    a0.text(-L / 2, -1.8, "carrier plate", ha="center", va="top", fontsize=8.5, color=GREEN)
    a0.text(L / 2, -1.8, "carrier plate", ha="center", va="top", fontsize=8.5, color=GREEN)
    a0.set_ylim(-4, 3.4); a0.axis("off")
    x = np.linspace(-L / 2, L / 2, 400)
    R = F / 2
    q = F / Bc
    s_ = x + L / 2
    a_ = (L - Bc) / 2
    l_ = np.clip(s_ - a_, 0, Bc)
    M = R * s_ - q * l_ * (s_ - a_ - l_ / 2)
    a1.fill_between(x, 0, M / 1000, color=C_PLANET); a1.plot(x, M / 1000, color=INK, lw=1.2)
    a1.annotate(f"$M_{{max}} = F_{{pin}}(L/4 - B_c/8)$ = {pin['M_max'] / 1000:.2f} N·m", xy=(0, pin["M_max"] / 1000), xytext=(-L / 2 + 0.5, pin["M_max"] / 1000 * 1.15),
                arrowprops=dict(arrowstyle="-", color=INK, lw=0.6), fontsize=9)
    a1.set_ylim(0, pin["M_max"] / 1000 * 1.3)
    a1.set_ylabel("bending moment [N·m]"); a1.set_xlabel("position along the pin [mm]")
    for sp in ("top", "right"):
        a1.spines[sp].set_visible(False)
    a0.set_title(f"Planet pin as a simply supported beam: Ø{pin['d']:g}, σ_b = {pin['sigma_b']:.0f} MPa, deflection {pin['deflection'] * 1000:.1f} µm", fontsize=10)
    plt.tight_layout()
_pin_figure()

$$M_{max} = F_{pin}\left(\frac{L}{4} - \frac{B_c}{8}\right),\qquad \sigma_b = \frac{32\,M_{max}}{\pi d_{pin}^3},\qquad \delta = \frac{F_{pin}\,(8L^3 - 4LB_c^2 + B_c^3)}{384\,E\,I},\qquad I = \frac{\pi d_{pin}^4}{64},$$

with $d_{pin} = F_w$, $E$ = 210 GPa, plate thickness $t_c = \max(4\ \text{mm},\ \lceil 0.5\,d_{pin}\rceil)$ unless you set it, and overall pin length $b + 2s_w + 2t_c$.

The pin load keeps a fixed direction relative to the carrier, so the pin stress is steady and the check is against the hardened pin's yield with a margin (default allowable 400 MPa for 100Cr6 at 60 HRC, which covers stress raisers at the plate edges). Deflection above about 0.01 mm per 10 mm of face width will push $K_{H\beta}$ above the default 1.2.

Pressure of the pin on each plate bore: $p = F_{pin}/(2\,d_{pin}\,t_c)$; keep it under about 150 MPa for a steel carrier. Pin fit in the plates: the pin is ground to one diameter (j5, for the bearing) and the plate holes are **N7**, which gives a light press fit of about 2–30 µm; secure the pin against turning (flat and plate, or a cross pin).

### Cantilevered Pin (Single-Plate Carrier)

With `carrier = "single"` there is one plate and each pin is pressed into it and overhangs, as in compact actuator gearboxes (motor-integrated, high ratio): with a high ratio the planets are so large that no posts fit between them. The pin is a cantilever, clamped over the plate thickness $t_c$ and loaded at the middle of the planet:

In [ ]:
#| echo: false
def _cantilever_figure():
    pl = _PS.planet
    d, b, sw, tc = 8.0, pl.b, 1.0, 10.0
    L = sw + b / 2
    fig, ax = plt.subplots(figsize=(11, 5.0))
    # plate
    ax.add_patch(Rectangle((-tc, -16), tc, 32, fc="#cfe8d6", ec=GREEN, lw=0.9, hatch="xx"))
    ax.text(-tc / 2, 17, f"carrier plate, $t_c$", ha="center", va="bottom", fontsize=9, color=GREEN)
    # pin (bent, exaggerated)
    xs = np.linspace(0, sw + b + sw + 1.5, 60)
    k = 0.012
    yb = -k * np.where(xs < L, xs ** 2 * (3 * L - xs) / (2 * L ** 3), (3 * xs - L) / 2 / L) * 40
    ax.add_patch(Rectangle((-tc, -d / 2), tc, d, fc=C_STEEL, ec=INK, lw=0.7))
    ax.fill_between(xs, yb - d / 2, yb + d / 2, color=C_STEEL, ec=INK, lw=0.7)
    # planet (tilted)
    ang = np.degrees(np.arctan(-k * 40 * 3 / (2 * L)))
    from matplotlib.transforms import Affine2D
    pr = Rectangle((sw, d / 2 + 0.6), b, 9, fc=C_PLANET, ec=INK, lw=0.8)
    pr.set_transform(Affine2D().rotate_deg_around(L, 0, ang) + ax.transData); ax.add_patch(pr)
    pr2 = Rectangle((sw, -d / 2 - 9.6), b, 9, fc=C_PLANET, ec=INK, lw=0.8)
    pr2.set_transform(Affine2D().rotate_deg_around(L, 0, ang) + ax.transData); ax.add_patch(pr2)
    ax.text(L + 4, 15.5, "planet (tilt exaggerated)", ha="center", fontsize=9)
    # load
    ax.annotate("", xy=(L + 0.5, -18), xytext=(L + 0.5, -11), arrowprops=dict(arrowstyle="-|>", color=RED, lw=1.4))
    ax.text(L + 1.1, -16, "$F_{pin}$ (tooth forces on the planet)", color=RED, fontsize=9)
    _dim(ax, (0, -20.5), (L, -20.5), "", color=BLUE)
    ax.text(L / 2, -22, "lever $L = s_w + b/2$", color=BLUE, ha="center", va="top", fontsize=9)
    _label(ax, "highest bore pressure at the plate face:\n$p_{max} = \\frac{F_{pin}}{d\\,t_c}\\left(4 + \\frac{6L}{t_c}\\right)$", (-0.3, -d / 2), (-tc - 26, -13), color=RED)
    # slope
    ax.plot([L - 6, L + 6], [0, 0], color=INK, lw=0.5, ls=":")
    ax.text(sw + b + sw + 3, 1, "slope $\\theta = F_{pin} L^2 / (2EI)$\ntilts the planet: $f_{ma} = \\theta\\,b$", fontsize=9, va="center")
    ax.set_xlim(-tc - 27, sw + b + sw + 24); ax.set_ylim(-25, 20); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Cantilevered planet pin in a single-plate carrier", fontsize=10)
    plt.tight_layout()
_cantilever_figure()

$$L = s_w + \frac{b}{2},\qquad M = F_{pin}\,L,\qquad \sigma_b = \frac{32\,M}{\pi d_{pin}^3},\qquad \delta = \frac{F_{pin}L^3}{3EI},\qquad \theta = \frac{F_{pin}L^2}{2EI}.$$

The pin seat in the plate carries the force and the moment; with a linear pressure along the seat the highest pressure is at the plate face:

$$p_{max} = \frac{F_{pin}}{d_{pin}\,t_c}\left(4 + \frac{6L}{t_c}\right) \le p_{allow},\qquad t_c = \max(4\ \text{mm},\ \lceil 1.2\,d_{pin}\rceil)\ \text{unless set}.$$

The slope tilts the planet against the sun and the ring, a misalignment $f_{ma} = \theta\,b$ across the face. It is turned into a face load factor with the mesh stiffness $c_\gamma \approx 20$ N/(mm·µm) (ISO 6336-1, $K_{H\beta} \le 2$):

$$K_{H\beta} = 1 + \frac{c_\gamma\,f_{ma}}{2\,w_m},\qquad w_m = \frac{K_A K_\gamma F_t}{b},$$

which must not exceed the $K_{H\beta}$ assumed in the rating (default 1.2); otherwise the pin is made thicker, or the planets need crowning or a lead correction. The slope is also held to 0.01 mm per 10 mm of face. The pin is $t_c + b + 2s_w + 1.5$ mm long, with a retaining ring at the free end; the plate holes are N7 (or P6 for a firmer seat), square to the plate face.

## Carrier: Theory

The plates must hold the pins with a ring of material around each hole, and pass inside the ring's tip circle during assembly. In the double-plate carrier, posts between the planets tie the two plates together (a single-plate carrier has no posts; there the planets only need a tip gap $2a_w\sin(\pi/N) - d_{a,p} \ge 0.5$ mm, and the axial stack has one plate, $t_c + b + 2s_w + 2c_{ax}$):

In [ ]:
#| echo: false
def _carrier_figure():
    ps, ca, pin = _PS, _CO.carrier, _CO.pin
    N, aw, Dc = ps.n_planets, ps.a_w, ca["plate_d"]
    fig, ax = plt.subplots(figsize=(11, 7.4))
    ax.add_patch(Circle((0, 0), ps.ring.da / 2, fill=False, ec=INK, ls=":", lw=0.8))
    ax.add_patch(Circle((0, 0), Dc / 2, fc="#e6f2ea", ec=GREEN, lw=1.1))
    ax.add_patch(Circle((0, 0), ca["sun_clearance_bore"] / 2, fc="white", ec=GREEN, lw=0.9))
    r_in = ca["sun_clearance_bore"] / 2
    for k in range(N):
        th = np.pi / 2 + 2 * np.pi * k / N
        c = aw * np.array([np.cos(th), np.sin(th)])
        ax.add_patch(Circle(c, ps.planet.da / 2, fill=False, ec=INK, ls="--", lw=0.6))
        ax.add_patch(Circle(c, pin["d"] / 2, fc="white", ec=GREEN, lw=0.9))
        tp = th + np.pi / N
        w = ca["post_width"]
        P = np.array([[r_in, -w / 2], [Dc / 2, -w / 2], [Dc / 2, w / 2], [r_in, w / 2]])
        ax.add_patch(Polygon(_rot(P, tp), closed=True, fc="#b9dcc4", ec=GREEN, lw=0.8, hatch="////"))
    ax.add_patch(Circle((0, 0), aw, fill=False, ec=RED, ls="-.", lw=0.7))
    c0 = np.array([0, aw])
    e = (Dc - 2 * aw - pin["d"]) / 2
    _dim(ax, c0 + [0, pin["d"] / 2], c0 + [0, pin["d"] / 2 + e], "", color=BLUE)
    tp = np.pi / 2 + np.pi / N
    u, v = np.array([np.cos(tp), np.sin(tp)]), np.array([-np.sin(tp), np.cos(tp)])
    rp = (Dc / 2 + r_in) / 2
    _dim(ax, rp * u - ca["post_width"] / 2 * v, rp * u + ca["post_width"] / 2 * v, "", color=BLUE)
    L = Dc / 2 + 18
    _label(ax, f"plate Ø$D_c$ = {Dc:g} (≤ ring tip Ø{ps.ring.da:.1f} − 2)", (Dc / 2 * np.cos(0.3), Dc / 2 * np.sin(0.3)), (L, 40), color=GREEN)
    _label(ax, "ring tip circle $d_{a,r}$ (the plate must pass inside)", (ps.ring.da / 2 * np.cos(0.05), ps.ring.da / 2 * np.sin(0.05)), (L, 22))
    _label(ax, f"pin circle Ø$2a_w$ = {2 * aw:g}", (aw * np.cos(-0.3), aw * np.sin(-0.3)), (L, -8), color=RED)
    _label(ax, f"post: width $w_{{post}} = 2a_w\\sin(\\pi/N) - d_{{a,p}} - 2$ = {ca['post_width']:.1f},\nradial height $h_{{post}}$ = {ca['post_height']:.1f}, centroid radius $r_{{post}}$",
           rp * u, (-L, 40), color=GREEN, ha="right")
    _label(ax, f"boss wall e ≥ max(3, 0.5 $d_{{pin}}$) around each pin hole\n(Ø{pin['d']:g} N7)", c0 + [0, pin["d"] / 2 + e / 2], (-L, 64), color=BLUE, ha="right")
    _label(ax, f"sun clearance bore Ø{ca['sun_clearance_bore']:.1f} = $d_{{a,s}}$ + 2", (r_in * np.cos(3.6), r_in * np.sin(3.6)), (-L, -40), color=GREEN, ha="right")
    _label(ax, "planet tip circles", c0 + ps.planet.da / 2 * np.array([np.cos(0.4), np.sin(0.4)]), (L, 62))
    ax.set_xlim(-L - 105, L + 110); ax.set_ylim(-Dc / 2 - 6, Dc / 2 + 6); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Carrier plate with {N} pins and {N} posts (example 18/27/72, m = 2)", fontsize=10)
    plt.tight_layout()
_carrier_figure()


$$2a_w + d_{pin} + 2e \le D_c \le d_{a,r} - 2c,$$

with $e \ge \max(3\text{ mm}, 0.5\,d_{pin})$ the boss wall and $c$ = 1 mm. The plate is made as large as the ring allows, $D_c = \lfloor d_{a,r} - 2 \rfloor$, which must not be below the left-hand side. Between the planets, $N$ posts tie the two plates together. A post has the tangential width left between neighbouring planet tips at radius $a_w$, $w_{post} = 2a_w\sin(\pi/N) - d_{a,p} - 2c$, and runs radially from the sun tip (plus clearance) to the plate edge. The free plate passes half the planet forces to the output plate through the posts, so each post is a beam fixed at both ends with a shear force

$$V = \frac{1000\,K_A\,T_c}{2N\,r_{post}},\qquad M = \frac{V\,L_{post}}{2},\qquad \sigma = \frac{6M}{h_{post}\,w_{post}^2},$$

with the post running from $r_{in} = d_{a,s}/2 + 1$ mm (clear of the sun) to the plate edge:

$$h_{post} = \frac{D_c}{2} - r_{in},\qquad r_{post} = \frac{D_c/2 + r_{in}}{2},\qquad L_{post} = b + 2s_w,$$

and the post width at least 3 mm. Keep $\sigma$ under about 120 MPa for a steel carrier (the post sees a pulsating load). The output shaft on the carrier plate needs $d \ge \sqrt[3]{16\,K_A T_c/(\pi\,\tau_{allow})}$, with $\tau_{allow}$ ≈ 40 MPa for a keyed steel shaft.

## Ring Fixing: Theory

The ring reaction torque $T_r = T_c - T_s$ goes into the housing. If $z_b$ bolts clamp the ring axially on a bolt circle $D_{bc}$, friction alone must carry it with a slip safety $S_{slip}$:

In [ ]:
#| echo: false
def _ring_figure():
    ps = _PS
    ci = _cmp.ComponentInputs(envelope_d=200, shaft_d=16)
    co = _cmp.size_components(ps, _RT, ci, life_h=10_000, n_in=1500.0)
    rm = co.ring_mount
    fig, ax = plt.subplots(figsize=(10, 7))
    R = rm["ring_od"] / 2
    ax.add_patch(Circle((0, 0), R, fc=C_RING, ec=INK, lw=0.9))
    ax.add_patch(Polygon(_rot(gear_outline(ps.ring), np.pi / 2), closed=True, fc="white", ec=INK, lw=0.7))
    ax.add_patch(Circle((0, 0), ps.ring.df / 2, fill=False, ec=BLUE, ls="--", lw=0.7))
    zb, Dbc, dh = rm["bolts"], rm["bolt_circle_d"], rm["clearance_hole"]
    ax.add_patch(Circle((0, 0), Dbc / 2, fill=False, ec=RED, ls="-.", lw=0.7))
    for j in range(zb):
        a = 2 * np.pi * (j + 0.5) / zb
        ax.add_patch(Circle(Dbc / 2 * np.array([np.cos(a), np.sin(a)]), dh / 2, fc="white", ec=INK, lw=0.7))
    a = 2 * np.pi * 0.5 / zb
    L = R + 16
    _label(ax, f"{zb} × {ci.bolt}-8.8 bolts, clearance hole Ø{dh:g}", Dbc / 2 * np.array([np.cos(a), np.sin(a)]), (L, 60))
    _label(ax, f"bolt circle $D_{{bc}}$ = OD − 2(1.5 + $d_h$/2) = Ø{Dbc:.1f}", Dbc / 2 * np.array([np.cos(0.0), np.sin(0.0)]), (L, 30), color=RED)
    _label(ax, f"ring OD = $D_{{env}} - 2t_{{housing}}$ = Ø{rm['ring_od']:g}", (R * np.cos(-0.3), R * np.sin(-0.3)), (L, 0))
    _label(ax, f"ring root circle $d_{{f,r}}$ = Ø{ps.ring.df:.1f};\nrim $s_R$ = (OD − $d_{{f,r}}$)/2 = {rm['ring_rim']:.1f}", (ps.ring.df / 2 * np.cos(-0.6), ps.ring.df / 2 * np.sin(-0.6)), (L, -35), color=BLUE)
    ax.text(0, 0, f"reaction torque\n$T_r$ = {rm['ring_torque']:.0f} N·m\nheld by friction under\nthe bolt clamp force", ha="center", va="center", fontsize=9)
    ax.set_xlim(-R - 10, L + 120); ax.set_ylim(-R - 6, R + 6); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"Ring clamped by bolts (example with a Ø{ci.envelope_d:g} envelope; a thinner rim is pressed in and doweled instead)", fontsize=10)
    plt.tight_layout()
_ring_figure()


$$z_b \ge \frac{2000\,S_{slip}\,K_A\,T_r}{\mu\,D_{bc}\,F_M}$$

with the assembly preload $F_M$ of the bolt (VDI 2230 Table A1, property class 8.8, thread friction 0.12, 90 % yield), $\mu$ = 0.15 between dry steel faces and $S_{slip}$ = 1.5, and never fewer bolts than planets. The ring outside diameter and rim are

$$D_{ring} = D_{env} - 2t_{housing},\qquad s_R = \frac{D_{ring} - d_{f,r}}{2} \ge 1.75\,m\ \text{(ISO 6336-3 limit)}.$$

Bolts through the ring need a rim wide enough for the clearance hole $d_h$ (ISO 273 medium) plus 1.5 mm of wall on each side, with the rim inside the bolt holes still at $3.5\,m$:

$$s_R \ge d_h + 2\cdot 1.5 + 3.5\,m,\qquad D_{bc} = D_{ring} - 2\,(1.5 + d_h/2).$$

When the rim is too thin for that, the ring is pressed into the housing (H7/s6 class) and pinned with dowels.

## Axial Stack

The length needed is two carrier plates, two thrust washers per planet, the face width, and the running clearances; the sun hub sits inside it. The axial envelope must hold

$$L_{stack} = 2t_c + b + 2s_w + 2c_{ax}.$$

The thrust washers are sized to sit on the pin and stay inside the planet root: inside diameter $F_w + 0.5$, outside diameter $\min(d_{f,p} - 1,\ E_w + 6)$, thickness $s_w$.

In [ ]:
#| echo: false
def _stack_figure():
    ps, ax_ = _PS, _CO.axial
    tc, sw, cax, b = _CO.carrier["plate_t"], _CI.washer_t, _CI.axial_clearance, ps.planet.b
    fig, ax = plt.subplots(figsize=(10, 4.4))
    x = 0.0
    parts = [("plate", tc, "#cfe8d6", GREEN), ("gap", cax, None, None), ("washer", sw, "#777", INK), ("planet", b, C_PLANET, INK),
             ("washer", sw, "#777", INK), ("gap", cax, None, None), ("plate", tc, "#cfe8d6", GREEN)]
    H = {"plate": 24, "washer": 12, "planet": 20}
    for nm, w, fc, ec in parts:
        if fc:
            ax.add_patch(Rectangle((x, 0), w, H[nm], fc=fc, ec=ec, lw=0.8, hatch="xx" if nm == "plate" else ("////" if nm == "planet" else None)))
        x += w
    stack = x
    ax.add_patch(Rectangle((-3, -4), stack + 6, 3, fc=C_STEEL, ec=INK, lw=0.6))
    ax.text(stack / 2, -2.5, "planet pin", ha="center", va="center", fontsize=8.5)
    xs = np.cumsum([0] + [w for _, w, _, _ in parts])
    labels = [f"$t_c$ = {tc:g}", f"$c_{{ax}}$ = {cax:g}", f"$s_w$ = {sw:g}", f"b = {b:g}", f"$s_w$ = {sw:g}", f"$c_{{ax}}$ = {cax:g}", f"$t_c$ = {tc:g}"]
    small = {1: (40, -6), 2: (38, 4), 4: (38, -4), 5: (40, 6)}   # label row and outward shift for thin parts
    for i, t in enumerate(labels):
        mid = (xs[i] + xs[i + 1]) / 2
        _dim(ax, (xs[i], 30), (xs[i + 1], 30), "", color=BLUE)
        if i in small:
            yl, dx = small[i]
            ax.annotate(t, xy=(mid, 30.5), xytext=(mid + dx, yl), ha="center", fontsize=9, color=BLUE,
                        arrowprops=dict(arrowstyle="-", color=BLUE, lw=0.6))
        elif i in (0, 6):   # plates: label outside the stack so the thin-part leaders stay clear
            ax.text(xs[0] - 1.5 if i == 0 else xs[-1] + 1.5, 30, t, ha="right" if i == 0 else "left", va="center", fontsize=9, color=BLUE)
        else:
            ax.text(mid, 32.4, t, ha="center", fontsize=9, color=BLUE)
    _dim(ax, (0, -9), (stack, -9), f"$L_{{stack}} = 2t_c + b + 2s_w + 2c_{{ax}}$ = {stack:g} mm (≤ axial envelope)", off=(0, -2.5), color=INK)
    ax.set_xlim(-16, stack + 16); ax.set_ylim(-15, 48); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Axial stack at a planet (half section)", fontsize=10)
    plt.tight_layout()
_stack_figure()

**Lubrication.** A first choice from the pitch line velocity $v$ (relative to the carrier):

| $v$ [m/s] | < 1.5 | 1.5 – 5 | 5 – 12 | ≥ 12 |
|---|---|---|---|---|
| lubricant | NLGI 00 semi-fluid grease or ISO VG 320 oil | ISO VG 220 EP oil, splash (or NLGI 00 grease, sealed for life) | ISO VG 150 EP oil, splash | ISO VG 100 EP oil, spray or forced |

**Overall result.** `PlanetaryComponents.ok` is true when every check passes: the sun joint (key or spline pressure; press-fit torque, bore stress and growth; adhesive torque; or shaft torsion), sun rim, bearing life, pin stress, plate pressure, pin deflection (single plate: pin slope and its $K_{H\beta}$), carrier plate inside the ring, post width and stress (single plate: planet tip gap), ring rim, and (if given) the axial length.

In [ ]:
#| export
# VDI 2230 Table A1 (approximate): 8.8, μ_G = 0.12 → (thread, assembly preload F_M [N], tightening torque M_A [N·m], clearance hole [mm])
BOLTS_88 = {"M3": (2_300, 1.3, 3.4), "M4": (4_000, 3.0, 4.5), "M5": (6_600, 5.9, 5.5), "M6": (9_300, 10.1, 6.6),
            "M8": (17_000, 24.6, 9.0), "M10": (27_000, 48.0, 11.0), "M12": (39_400, 84.0, 13.5)}


@dataclass(frozen=True)
class ComponentInputs:
    """
    Envelope and choices for the parts around the gears. Lengths in mm.

    envelope_d       : outside diameter available [mm]
    envelope_length  : axial length available for the stage [mm] (None: not checked)
    shaft_d          : input shaft diameter in the sun bore [mm]
    hub_joint        : 'key' (DIN 6885-1), 'spline' (DIN 5480), 'press' (interference fit), 'adhesive'
                       (retaining compound) or 'integral' (sun cut on the shaft; shaft_d is then the shaft beside the sun)
    spline_module    : module of the spline if hub_joint == 'spline'
    spline_fit       : 'sliding' (9H/8f, backlash, floating sun) or 'tight' (9H/8k, no backlash)
    press_fit        : fit for 'press' (e.g. 'H6/s5'); None: lightest of PRESS_FITS that passes
    mu_fit, Rz_fit   : friction coefficient and surface roughness Rz [µm] of the press fit
    slip_safety      : required torque capacity of press fit or adhesive over K_A·T_sun
    sigma_hub_allow  : allowable equivalent stress at the sun bore [MPa]
    tau_adhesive_allow : design shear stress of the retaining compound [MPa]
    shrink_T_max     : highest temperature the hardened sun may be heated to for shrinking [°C]
    hob_d            : hob tip diameter, for the run-out of a sun cut on its shaft [mm]
    carrier          : 'double' (two plates joined by posts, pins supported at both ends) or 'single'
                       (one plate, cantilevered pins, no posts between the planets)
    c_gamma          : mesh stiffness [N/(mm·µm)] for the face load factor from the pin slope ('single')
    housing_wall     : housing wall outside the ring (0: the ring is the outer wall)
    planet_rim       : rim over the planet bore, in tooth depths (1.2: no rim penalty)
    bearing_life_h   : required L10h of the planet bearings (None: same as the gear life)
    plate_t          : carrier plate thickness (None: max(4, ceil(0.5·d_pin)) double, max(4, ceil(1.2·d_pin)) single)
    washer_t         : thrust washer thickness per side
    axial_clearance  : running clearance per side
    bolt             : bolt size for clamping the ring ('M3' … 'M12')
    p_key_allow, sigma_pin_allow, p_plate_allow, sigma_post_allow, tau_shaft_allow : allowables [MPa]
    """
    envelope_d: float
    shaft_d: float
    envelope_length: float | None = None
    hub_joint: str = "key"
    spline_module: float = 1.0
    spline_fit: str = "sliding"
    press_fit: str | None = None
    mu_fit: float = 0.12
    Rz_fit: float = 2.0
    slip_safety: float = 1.5
    sigma_hub_allow: float = 500.0
    tau_adhesive_allow: float = 8.0
    shrink_T_max: float = 150.0
    hob_d: float = 40.0
    carrier: str = "double"
    c_gamma: float = 20.0
    housing_wall: float = 0.0
    planet_rim: float = 1.2
    bearing_life_h: float | None = None
    plate_t: float | None = None
    washer_t: float = 1.0
    axial_clearance: float = 0.5
    bolt: str = "M5"
    p_key_allow: float = 150.0
    sigma_pin_allow: float = 400.0
    p_plate_allow: float = 150.0
    sigma_post_allow: float = 120.0
    tau_shaft_allow: float = 40.0


def _hub_depth(ci: ComponentInputs) -> float:
    if ci.hub_joint == "key":
        return ParallelKey.for_shaft(ci.shaft_d).t2
    if ci.hub_joint == "spline":
        return 0.55 * ci.spline_module + 1.1 * ci.spline_module   # spline root below the nominal bore
    return 0.0


def _cage_for_bore(Ew: float) -> tuple[float, float]:
    "Pin diameter F_w and roller diameter D_w for a planet bore E_w [mm]."
    Dw = needle_roller_diameter(Ew - 5)
    Fw = Ew - 2 * Dw
    for _ in range(3):                       # settle the roller size for the pin actually found
        Dw = needle_roller_diameter(Fw)
        Fw = Ew - 2 * Dw
    return Fw, Dw


def max_planet_bore(ps: PlanetarySet, ci: ComponentInputs) -> float:
    "Largest whole-mm planet bore E_w [mm] that keeps the planet rim of `ci.planet_rim` tooth depths."
    p = ps.planet
    return float(math.floor(p.df - 2 * ci.planet_rim * (p.ha + p.hf) * p.m))


def rim_thicknesses(ps: PlanetarySet, ci: ComponentInputs, planet_bore: float | None = None) -> dict[str, float | None]:
    """
    Rim under the tooth roots for the rim factor Y_B [mm]: sun over the hub groove, planet over its
    bore (default: the largest bore allowed), ring outside its roots.
    """
    t2 = _hub_depth(ci)
    sun_rim = None if ci.hub_joint == "integral" else ps.sun.df / 2 - ci.shaft_d / 2 - t2
    Ew = max_planet_bore(ps, ci) if planet_bore is None else planet_bore
    ring_od = ci.envelope_d - 2 * ci.housing_wall
    return {"sun": sun_rim, "planet": (ps.planet.df - Ew) / 2, "ring": (ring_od - ps.ring.df) / 2}



In [ ]:
#| export
@dataclass(frozen=True)
class Check:
    "One component check: value against a limit."
    name: str
    value: float
    limit: float
    unit: str
    upper: bool = True          # True: value must not exceed limit

    @property
    def ok(self) -> bool:
        return self.value <= self.limit + 1e-9 if self.upper else self.value >= self.limit - 1e-9

    def __str__(self) -> str:
        rel = "≤" if self.upper else "≥"
        return f"{'✓' if self.ok else '✗'} {self.name}: {self.value:.3g} {self.unit} (need {rel} {self.limit:.3g})"


@dataclass(frozen=True)
class PlanetaryComponents:
    """
    Dimensions of the parts around the gears (mm, N, N·m) and their checks.

    sun_hub    : dict  joint type, key size and length or spline data, rim
    bearing    : dict  cage assembly Fw × Ew × Bc, rollers, load, speed, C_req, C_est, L10h
    pin        : dict  diameter, length, span, stress, deflection, fits
    carrier    : dict  plate diameter and thickness, pin circle, posts, output shaft
    ring_mount : dict  ring OD, bolts or press fit, torque
    axial      : dict  stack length and its parts
    lubrication: str   starting recommendation
    checks     : tuple[Check, ...]
    """
    sun_hub: dict
    bearing: dict
    pin: dict
    carrier: dict
    ring_mount: dict
    axial: dict
    lubrication: str
    checks: tuple[Check, ...]

    @property
    def ok(self) -> bool:
        return all(c.ok for c in self.checks)

    def problems(self) -> list[str]:
        return [str(c) for c in self.checks if not c.ok]


def lubrication_advice(v: float) -> str:
    "Starting lubricant choice from the pitch line velocity v [m/s] (AGMA 9005 style ranges)."
    if v < 1.5:
        return f"v = {v:.1f} m/s: semi-fluid grease NLGI 00 (EP, synthetic base) or ISO VG 320 oil"
    if v < 5:
        return f"v = {v:.1f} m/s: ISO VG 220 EP gear oil, splash lubrication, or NLGI 00 grease for sealed-for-life units"
    if v < 12:
        return f"v = {v:.1f} m/s: ISO VG 150 EP gear oil, splash lubrication; grease not recommended"
    return f"v = {v:.1f} m/s: ISO VG 100 EP gear oil, spray or forced lubrication"


def size_components(ps: PlanetarySet, rating: PlanetaryRating, ci: ComponentInputs,
                    life_h: float, n_in: float, K_A: float = 1.25, density: float = 7.85e-6) -> PlanetaryComponents:
    """
    Size the sun joint, planet bearings, pins, carrier and ring fixing (see the theory sections).

    life_h, n_in, K_A: as used for the rating; density in kg/mm³ for the planet centrifugal force.
    """
    checks: list[Check] = []
    sun, pl, ring = ps.sun, ps.planet, ps.ring
    N = ps.n_planets
    rims = rim_thicknesses(ps, ci)
    sp = ps.speeds(n_in)
    if rating.factors.get("K_gamma") is None:
        raise ValueError("rating has no K_gamma")

    # --- sun hub -----------------------------------------------------------------
    T_s = K_A * rating.T_sun
    L_j = sun.b                                           # joint length = sun width
    hub: dict = {"joint": ci.hub_joint, "shaft_d": ci.shaft_d, "rim": rims["sun"]}
    if ci.hub_joint == "key":
        key = ParallelKey.for_shaft(ci.shaft_d)
        L_key = ParallelKey.standard_length(sun.b + 2 * ci.washer_t)
        p_key = key.pressure(T_s, L_key)
        hub.update(bore_fit="H7 (shaft k6)", backlash="key side clearance (P9 groove keeps it small)",
                   key=f"DIN 6885-1 A {key.b:g} × {key.h:g} × {L_key:g}", key_b=key.b, key_h=key.h,
                   t1=key.t1, t2=key.t2, key_length=L_key, key_pressure=p_key,
                   hub_groove_width="P9 (fixed key) or JS9", hub_groove_depth=f"{ci.shaft_d + key.t2:.1f} +0.1/0 over bore")
        checks.append(Check("key flank pressure", p_key, ci.p_key_allow, "MPa"))
    elif ci.hub_joint == "spline":
        if ci.spline_fit not in SPLINE_FITS:
            raise ValueError(f"spline_fit must be one of {list(SPLINE_FITS)}")
        g_sp = spline_geometry(ci.shaft_d, ci.spline_module)
        p_sp = spline_pressure(T_s, ci.shaft_d, ci.spline_module, L_j)
        fit_txt, note = SPLINE_FITS[ci.spline_fit]
        hub.update(bore_fit=fit_txt, backlash=note,
                   spline=f"DIN 5480 W {ci.shaft_d:g} × {ci.spline_module:g} × {g_sp['z']} × {fit_txt}",
                   spline_z=g_sp["z"], spline_pressure=p_sp)
        checks.append(Check("spline flank pressure", p_sp, ci.p_key_allow, "MPa"))
    elif ci.hub_joint == "press":
        T_req = ci.slip_safety * T_s
        pf = (press_fit(ci.shaft_d, sun.df, L_j, ci.press_fit, ci.mu_fit, ci.Rz_fit) if ci.press_fit
              else select_press_fit(ci.shaft_d, sun.df, L_j, T_req, ci.sigma_hub_allow, ci.mu_fit, ci.Rz_fit))
        a_w, aw = ps.a_w, ps.sun_planet.alpha_w
        j_min = 2 / 3 * (0.06 + 0.0005 * a_w + 0.03 * sun.m)
        hub.update(bore_fit=pf.fit, backlash="none", press_fit=pf.fit,
                   interference_um=f"{pf.delta_min * 1000:.1f} … {pf.delta_max * 1000:.1f} (effective)",
                   p_min=pf.p_min, p_max=pf.p_max, T_slip=pf.T_slip, sigma_hub=pf.sigma_hub, hoop_stress_at_root=pf.sigma_root,
                   sun_growth_um=pf.growth * 1000, shrink_temperature=pf.T_join, press_force_N=pf.F_press,
                   assembly=(f"shrink: heat the sun to {pf.T_join:.0f} °C" if pf.T_join <= ci.shrink_T_max else
                             f"heat the sun to {ci.shrink_T_max:g} °C and cool the shaft (dry ice or liquid nitrogen)")
                            + "; do not press through the motor bearings")
        checks.append(Check("press fit slip torque / (S·K_A·T_sun)", pf.T_slip / T_req, 1.0, "", upper=False))
        checks.append(Check("press fit stress at the sun bore", pf.sigma_hub, ci.sigma_hub_allow, "MPa"))
        checks.append(Check("backlash taken by sun growth / j_bn,min", pf.growth * math.sin(aw) / j_min, 0.5, ""))
    elif ci.hub_joint == "adhesive":
        T_cap = adhesive_torque(ci.shaft_d, L_j, ci.tau_adhesive_allow)
        hub.update(bore_fit="H7 (shaft k6) with anaerobic retaining compound", backlash="none",
                   adhesive_torque=T_cap, adhesive_note="clean and activate both surfaces; service below about 150 °C")
        checks.append(Check("adhesive torque / (S·K_A·T_sun)", T_cap / (ci.slip_safety * T_s), 1.0, "", upper=False))
    elif ci.hub_joint == "integral":
        tau = 16 * 1000 * T_s / (math.pi * sun.df ** 3)
        L_ro = hob_runout(ci.shaft_d / 2 - (sun.df / 2 - 0.25 * sun.m), ci.hob_d)
        hub.update(bore_fit="none: teeth cut on the shaft", backlash="none", shaft_torsion=tau,
                   journal_d=ci.shaft_d, hob_runout=L_ro,
                   note=("shaft beside the sun is below the root: the hob runs out freely" if L_ro == 0 else
                         f"shaft beside the sun is above the root: leave {L_ro:.1f} mm hob run-out (or shape the teeth)")
                        + "; connect the motor with a clamp hub or a torsionally stiff coupling")
        checks.append(Check("torsion in the sun shaft at the root diameter", tau, ci.tau_shaft_allow, "MPa"))
    else:
        raise ValueError("hub_joint must be 'key', 'spline', 'press', 'adhesive' or 'integral'")
    if rims["sun"] is not None:
        checks.append(Check("sun rim over hub groove / tooth depth", rims["sun"] / ((sun.ha + sun.hf) * sun.m), 0.5, "", upper=False))

    # --- planet bearing and pin: smallest cage assembly that passes ---------------------
    Bc = math.floor(pl.b - 1.0) if pl.b > 6 else pl.b
    a1, a2 = ps.sun_planet.alpha_w, ps.planet_ring.alpha_w
    Kg = rating.factors["K_gamma"]
    F_teeth = Kg * rating.F_bn * math.hypot(math.cos(a1) + math.cos(a2), math.sin(a1) - math.sin(a2))
    w_c = 2 * math.pi * sp["carrier"] / 60
    n_b = abs(sp["planet_rel"])
    L_h = ci.bearing_life_h or life_h
    E = 210_000.0
    Ew_max = max_planet_bore(ps, ci)
    if ci.carrier not in ("double", "single"):
        raise ValueError("carrier must be 'double' or 'single'")
    single = ci.carrier == "single"
    K_Hb_assumed = rating.factors.get("K_Hbeta", 1.2)
    trial = None
    for Ew in [float(x) for x in range(8, int(Ew_max) + 1)] or [Ew_max]:
        Fw, Dw = _cage_for_bore(Ew)
        if Fw < 4:
            continue
        m_p = density * math.pi / 4 * (pl.d ** 2 - Ew ** 2) * pl.b              # kg
        F_cf = m_p * w_c ** 2 * ps.a_w / 1000                                   # N
        F_pin = F_teeth + F_cf
        C_est, Z = needle_cage_rating(Fw, Dw, Bc)
        L10h = (C_est / F_pin) ** (10 / 3) * 1e6 / (60 * n_b) if n_b > 0 else math.inf
        I = math.pi * Fw ** 4 / 64
        if single:
            t_c = ci.plate_t if ci.plate_t is not None else max(4.0, math.ceil(1.2 * Fw))
            L = ci.washer_t + pl.b / 2                       # plate face to the middle of the planet
            M = F_pin * L
            delta = F_pin * L ** 3 / (3 * E * I)
            slope = F_pin * L ** 2 / (2 * E * I)             # rad, at the middle of the planet
            p_plate = F_pin / (Fw * t_c) * (4 + 6 * L / t_c)
            w_m = K_A * Kg * rating.F_t / pl.b
            K_Hb = 1 + ci.c_gamma * slope * pl.b * 1000 / (2 * w_m)
            tilt = slope * 10
        else:
            t_c = ci.plate_t if ci.plate_t is not None else max(4.0, math.ceil(0.5 * Fw))
            L = pl.b + 2 * ci.washer_t + t_c
            M = F_pin * (L / 4 - Bc / 8)
            delta = F_pin * (8 * L ** 3 - 4 * L * Bc ** 2 + Bc ** 3) / (384 * E * I)
            p_plate = F_pin / (2 * Fw * t_c)
            K_Hb = 1.0
            tilt = delta / pl.b * 10
        sig_pin = 32 * M / (math.pi * Fw ** 3)
        trial = (Ew, Fw, Dw, Z, F_pin, F_cf, C_est, L10h, t_c, L, M, sig_pin, delta, p_plate, K_Hb, tilt)
        if (L10h >= L_h and sig_pin <= ci.sigma_pin_allow and tilt <= 0.01
                and p_plate <= ci.p_plate_allow and K_Hb <= K_Hb_assumed):
            break
    if trial is None:
        raise ValueError("planet is too small for any needle cage assembly")
    Ew, Fw, Dw, Z, F_pin, F_cf, C_est, L10h, t_c, L, M, sig_pin, delta, p_plate, K_Hb, tilt = trial
    C_req = F_pin * (60 * n_b * L_h / 1e6) ** 0.3
    cage_txt = f"needle roller cage assembly K {Fw:g} × {Ew:g} × {Bc:g} (ISO 3030), or equivalent"
    if Bc > 2 * Fw:
        cage_txt += f"; if no single cage this wide is listed, use two cages of about {Bc / 2:.0f} mm side by side"
    brg = {"type": cage_txt,
           "Fw": Fw, "Ew": Ew, "Bc": Bc, "Dw": Dw, "rollers": Z, "F_pin": F_pin, "F_centrifugal": F_cf,
           "speed_rpm": n_b, "C_req": C_req, "C_est": C_est, "L10h_est": L10h, "planet_rim": (pl.df - Ew) / 2,
           "raceways": "pin and planet bore 58–64 HRC, Ra ≤ 0.2 µm, cylindricity IT3", "bore_fit": "G6", "pin_fit": "j5"}
    checks.append(Check("planet bearing L10h (estimated rating)", L10h, L_h, "h", upper=False))
    if single:
        pin_len = t_c + pl.b + 2 * ci.washer_t + 1.5                        # + retaining ring groove at the free end
        pin = {"d": Fw, "length": pin_len, "lever": L, "M_max": M, "sigma_b": sig_pin, "deflection": delta,
               "slope_per_10mm": tilt, "K_Hbeta_from_slope": K_Hb, "plate_pressure": p_plate,
               "material": "100Cr6 (1.3505) through hardened 58–64 HRC, or 16MnCr5 case hardened",
               "fit_in_plates": f"pressed into the plate over {t_c:g} mm: hole N7 (or P6) on the j5 pin; retaining ring at the free end",
               "raceway_fit": "j5", "surface": "Ra ≤ 0.2 µm on raceway"}
        checks += [Check("pin bending stress (cantilever)", sig_pin, ci.sigma_pin_allow, "MPa"),
                   Check("pin edge pressure in plate bore", p_plate, ci.p_plate_allow, "MPa"),
                   Check("pin slope per 10 mm face", tilt, 0.01, "mm"),
                   Check("face load factor from pin slope vs rating K_Hβ", K_Hb, K_Hb_assumed, "")]
    else:
        pin_len = pl.b + 2 * ci.washer_t + 2 * t_c
        pin = {"d": Fw, "length": pin_len, "span": L, "M_max": M, "sigma_b": sig_pin, "deflection": delta,
               "plate_pressure": p_plate, "material": "100Cr6 (1.3505) through hardened 58–64 HRC, or 16MnCr5 case hardened",
               "fit_in_plates": "plate holes N7 on the j5 pin (light press), pin secured against rotation", "raceway_fit": "j5", "surface": "Ra ≤ 0.2 µm on raceway"}
        checks += [Check("pin bending stress", sig_pin, ci.sigma_pin_allow, "MPa"),
                   Check("pin pressure on plate bore", p_plate, ci.p_plate_allow, "MPa"),
                   Check("pin deflection per 10 mm face", delta / pl.b * 10, 0.01, "mm")]

    # --- carrier ----------------------------------------------------------------------
    e = max(3.0, 0.5 * Fw)
    Dc_min = 2 * ps.a_w + Fw + 2 * e
    Dc_max = ring.da - 2.0
    Dc = math.floor(Dc_max) if Dc_max >= Dc_min else Dc_min
    w_post = 2 * ps.a_w * math.sin(math.pi / N) - pl.da - 2.0
    r_in = sun.da / 2 + 1.0
    h_post = Dc / 2 - r_in
    r_post = (Dc / 2 + r_in) / 2
    L_post = pl.b + 2 * ci.washer_t
    T_c = rating.T_carrier
    sig_post = math.inf
    if w_post > 0 and h_post > 0:
        V = K_A * T_c * 1000 / (2 * N * r_post)
        sig_post = 6 * (V * L_post / 2) / (h_post * w_post ** 2)
    d_out = (16 * K_A * T_c * 1000 / (math.pi * ci.tau_shaft_allow)) ** (1 / 3)
    carrier = {"type": ("single-plate carrier, cantilevered pins, output from the plate" if single else
                        "double-plate carrier with posts, output from one plate"),
               "plate_d": Dc, "plate_t": t_c, "pin_circle_d": 2 * ps.a_w, "pin_holes": f"{N} × Ø{Fw:g} N7 at {360 / N:g}°",
               "post_width": w_post, "post_height": h_post, "post_length": L_post, "post_stress": sig_post,
               "sun_clearance_bore": 2 * r_in, "output_shaft_min_d": d_out,
               "material": "42CrMo4 Q&T (1.7225) or C45 normalized, or sintered steel for series",
               "pin_hole_position": "pin circle radius ±(f_a) and angular position, see manufacturing data"}
    checks.append(Check("carrier plate inside ring tip circle", Dc_min, Dc_max, "mm"))
    if single:
        for k_ in ("post_width", "post_height", "post_length", "post_stress"):
            carrier.pop(k_)
        checks.append(Check("tip gap between neighbouring planets", 2 * ps.a_w * math.sin(math.pi / N) - pl.da, 0.5, "mm", upper=False))
    else:
        checks += [Check("carrier post width between planets", w_post, 3.0, "mm", upper=False),
                   Check("carrier post bending stress", sig_post, ci.sigma_post_allow, "MPa")]

    # --- ring fixing --------------------------------------------------------------------
    ring_od = ci.envelope_d - 2 * ci.housing_wall
    F_M, M_A, d_h = BOLTS_88[ci.bolt]
    T_r = abs(rating.T_ring)
    rim_w = (ring_od - ring.df) / 2
    D_bc = (ring_od + ring.df) / 2 + (d_h / 2 if rim_w > 0 else 0) * 0
    need_rim = d_h + 2 * 1.5 + 3.5 * ring.m
    if rim_w >= need_rim:
        D_bc = ring_od - 2 * (1.5 + d_h / 2)
        z_b = max(N, math.ceil(2000 * 1.5 * K_A * T_r / (0.15 * D_bc * F_M)))
        mount = {"method": f"{z_b} × {ci.bolt}-8.8 through bolts on Ø{D_bc:.1f} bolt circle, clamped between housing parts",
                 "bolts": z_b, "bolt_circle_d": D_bc, "tightening_torque": M_A, "clearance_hole": d_h,
                 "dowels": "2 × ISO 8734 dowel pins for location"}
    else:
        mount = {"method": "ring pressed into housing (H7/s6) and secured with dowel pins; rim too thin for bolt holes",
                 "bolts": 0}
    mount.update(ring_od=ring_od, ring_rim=rim_w, ring_torque=T_r)
    checks.append(Check("ring rim / module (ISO 6336-3 lower limit; Y_B > 1 below 3.5)", rim_w / ring.m, 1.75, "", upper=False))
    if ci.housing_wall == 0 and ring_od > ci.envelope_d + 1e-9:
        checks.append(Check("ring OD within envelope", ring_od, ci.envelope_d, "mm"))

    # --- axial stack ----------------------------------------------------------------------
    n_pl = 1 if single else 2
    stack = n_pl * t_c + pl.b + 2 * ci.washer_t + 2 * ci.axial_clearance
    axial = {"plates": n_pl * t_c, "face_width": pl.b, "washers": 2 * ci.washer_t,
             "clearances": 2 * ci.axial_clearance, "stack": stack,
             "thrust_washers": f"2 per planet, Ø{Fw + 0.5:.1f}/Ø{min(pl.df - 1, Ew + 6):.1f} × {ci.washer_t:g}, hardened steel or PEEK"}
    if ci.envelope_length is not None:
        checks.append(Check("axial stack within envelope length", stack, ci.envelope_length, "mm"))

    return PlanetaryComponents(hub, brg, pin, carrier, mount, axial, lubrication_advice(rating.v), tuple(checks))

### Example

The rated set from `gears.rating` (18/27/72, module 1.5, 20 mm face) in a 120 mm envelope on a 14 mm keyed shaft:

In [ ]:
from emachines.gears.layout import PlanetarySet
from emachines.gears.rating import RatingInputs, rate_planetary
ps = PlanetarySet.build(18, 27, 72, m=1.5, n_planets=3, b=20)
ci = ComponentInputs(envelope_d=120, shaft_d=14, envelope_length=40)
inp = RatingInputs(torque_out=40.0, n_in=3000.0)
r0 = rate_planetary(ps, inp, rims=rim_thicknesses(ps, ci))              # pass 1: largest planet bore
comp0 = size_components(ps, r0, ci, life_h=inp.life_h, n_in=inp.n_in)
rims = rim_thicknesses(ps, ci, planet_bore=comp0.bearing["Ew"])          # pass 2: the bore actually chosen
print({k: (round(v, 2) if v is not None else None) for k, v in rims.items()})
r = rate_planetary(ps, inp, rims=rims)
comp = size_components(ps, r, ci, life_h=inp.life_h, n_in=inp.n_in)
for name in ("sun_hub", "bearing", "pin", "carrier", "ring_mount", "axial"):
    d = getattr(comp, name)
    print(name + ":", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items()})
print(comp.lubrication)
for c in comp.checks:
    print(c)

The stage is 51 mm long against a 40 mm envelope, so this face width does not fit; `gears.design` searches for a set that does. The rating is run twice because the planet rim depends on the bearing, which depends on the load. The sun rim over the key groove is thin here, so the sun's bending stress carries a rim factor above 1:

In [ ]:
sun_r = next(g for g in r.gears if g.gear == "sun")
print(f"sun Y_B = {sun_r.Y_B:.3f}, S_F = {sun_r.S_F:.2f}")
assert sun_r.Y_B > 1.0

---

## Tests

In [ ]:
#| hide
# pin load: twice the line-of-action force when both meshes have the same α_w (ignoring centrifugal force)
assert math.isclose(comp.bearing["F_pin"] - comp.bearing["F_centrifugal"], 2 * r.F_bn * math.cos(ps.sun_planet.alpha_w) * r.factors["K_gamma"], rel_tol=1e-9)
# bearing geometry
b = comp.bearing
assert math.isclose(b["Ew"], b["Fw"] + 2 * b["Dw"]) and b["Ew"] <= ps.planet.df - 2 * 1.2 * 2.25 * 1.5
assert b["L10h_est"] >= 10_000
# carrier plate fits inside the ring tip circle and holds the pins
assert comp.carrier["plate_d"] <= ps.ring.da - 2 and comp.carrier["plate_d"] >= 2 * ps.a_w + b["Fw"]
# more torque needs at least as large a bearing
r_hi = rate_planetary(ps, RatingInputs(torque_out=200.0, n_in=3000.0), rims=rims)
comp_hi = size_components(ps, r_hi, ci, life_h=10_000, n_in=3000.0)
assert comp_hi.bearing["F_pin"] > comp.bearing["F_pin"] and comp_hi.bearing["Ew"] >= comp.bearing["Ew"]
# keys: every listed shaft diameter maps to a key
for d in (7, 9, 11, 14, 20, 25, 35, 40, 48, 55, 60, 70, 80, 90, 100):
    ParallelKey.for_shaft(d)